In [ ]:
!pip install seisbench
import seisbench
import seisbench.models as sbm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
import sys
import os
from scipy.signal import resample
try:
    import obspy

    obspy.read()
except TypeError:
    # Needs to restart the runtime once, because obspy only works properly after restart.
    print(
        "Stopping RUNTIME. If you run this code for the first time, this is expected. Colaboratory will restart automatically. Please run again."
    )
    exit()

In [ ]:
# Mostra tutti i dataset disponibili
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Functions import

In [ ]:
# Download utils from GitHub
url = 'https://raw.githubusercontent.com/giulianaparadiso99/tesi-seismic-analysis/main/src/phasenet_utils.py'
urllib.request.urlretrieve(url, 'phasenet_utils.py')
from phasenet_utils import get_station_from_filename, get_component_from_filename, create_obspy_stream_from_dataframe


## Data loading

In [ ]:
# Load data
df_signals = pd.read_parquet('/kaggle/input/datasets/giulianaparadiso/seismic-signals/acc_preprocessed_scaling.parquet')
df_meta = pd.read_parquet('/kaggle/input/datasets/giulianaparadiso/seismic-signals/metadata_clean.parquet')

# Loading PhaseNet model 

In [ ]:
model = sbm.PhaseNet.from_pretrained("instance")

## With resampling

In [ ]:
results = []
sampling_rate_original = 200  # Hz
sampling_rate_target = 100    # Hz for PhaseNet

for station_code in df_signals['file'].apply(get_station_from_filename).unique():
    mask = df_signals['file'].apply(lambda f: get_station_from_filename(f) == station_code)
    df_station = df_signals[mask].copy()
    
    # Create ObsPy Stream
    stream, comp_names = create_obspy_stream_from_dataframe(
        df_station, station_code, sampling_rate_original
    )
    
    if stream is None:
        print(f"Skipping {station_code}: incomplete components")
        continue
    
    original_starttime = stream[0].stats.starttime
    original_npts = stream[0].stats.npts
    original_duration = original_npts / sampling_rate_original
    
    # PhaseNet expects ~30s windows at 100 Hz
    expected_samples_100hz = 3001
    expected_duration = expected_samples_100hz / sampling_rate_target
    
    if original_duration < expected_duration:
        print(f"Skipping {station_code}: signal too short ({original_duration:.1f}s < {expected_duration:.1f}s)")
        continue
    
    # Resample to 100 Hz
    stream_resampled = stream.copy()
    stream_resampled.resample(sampling_rate_target)
    
    resampled_duration = stream_resampled[0].stats.npts / sampling_rate_target
    duration_diff = abs(original_duration - resampled_duration)
    if duration_diff > 0.5:
        print(f"Warning {station_code}: resampling changed duration by {duration_diff:.2f}s")
    
    # Apply PhaseNet
    annotations = model.annotate(stream_resampled)
    
    # Extract P and S probability traces
    p_traces = annotations.select(channel="PhaseNet_P")
    s_traces = annotations.select(channel="PhaseNet_S")
    
    if len(p_traces) == 0 or len(s_traces) == 0:
        print(f"Skipping {station_code}: no annotations produced")
        continue
    
    p_trace = p_traces[0]
    s_trace = s_traces[0]
    
    # Get probability arrays
    p_prob = p_trace.data
    s_prob = s_trace.data
    
    # Find onset as maximum probability
    p_idx_resampled = p_prob.argmax()
    s_idx_resampled = s_prob.argmax()
    
    # Time relative to annotation trace start
    t_p_annotation = p_idx_resampled / sampling_rate_target
    t_s_annotation = s_idx_resampled / sampling_rate_target
    
    time_offset = float(p_trace.stats.starttime - original_starttime)
    
    # Time relative to ORIGINAL stream start
    t_p_original = t_p_annotation + time_offset
    t_s_original = t_s_annotation + time_offset
    
    p_idx_original = int(round(t_p_original * sampling_rate_original))
    s_idx_original = int(round(t_s_original * sampling_rate_original))
    
    if p_idx_original < 0 or p_idx_original >= original_npts:
        print(f"Warning {station_code}: P pick outside signal bounds ({p_idx_original}/{original_npts})")
    if s_idx_original < 0 or s_idx_original >= original_npts:
        print(f"Warning {station_code}: S pick outside signal bounds ({s_idx_original}/{original_npts})")
    
    # Save results
    results.append({
        'station_code': station_code,
        'components': ', '.join(comp_names),
        't_p_samples': p_idx_original,
        't_s_samples': s_idx_original,
        't_p_seconds': float(t_p_original),
        't_s_seconds': float(t_s_original),
        'p_probability_max': float(p_prob[p_idx_resampled]),
        's_probability_max': float(s_prob[s_idx_resampled]),
        'time_offset_seconds': float(time_offset),
        'original_duration_s': float(original_duration),
        'annotation_npts': len(p_prob)
    })
    
    print(f"{station_code}: P={t_p_original:.2f}s, S={t_s_original:.2f}s, offset={time_offset:.2f}s")

# Convert to DataFrame
df_picks = pd.DataFrame(results)

# Check S > P 
invalid_picks = df_picks[df_picks['t_s_seconds'] <= df_picks['t_p_seconds']]
if len(invalid_picks) > 0:
    print(f"\nWARNING: {len(invalid_picks)} stations have S before or at P:")
    print(invalid_picks[['station_code', 't_p_seconds', 't_s_seconds']])

# Save
df_picks.to_csv('phasenet_picks_instance_corrected.csv', index=False)
print(f"\nProcessed {len(results)} stations")

In [ ]:
# ETHZ results (from previous CSV)
ethz_data = {
    'station_code': ['EILF', 'ESCA', 'ISO', 'MFC', 'MON', 'MVIF', 'OGAG', 'OGDI', 'OGMO', 'OGS3', 'OGSA', 'REVF', 'SAOF', 'SPIF', 'BHB', 'BRZ', 'CRI', 'SAV', 'SLZ', 'CAGN', 'MENA'],
    't_p_seconds': [60.65, 61.88, 62.76, 61.48, 60.84, 61.4, 62.68, 62.52, 61.29, 60.25, 61.48, 61.38, 61.96, 62.18, 20.34, 12.19, 26.66, 26.47, 26.48, 61.56, 61.53],
    't_s_seconds': [78.38, 72.71, 67.51, 68.6, 218.85, 70.79, 68.15, 27.09, 228.69, 72.55, 15.7, 74.09, 71.58, 69.51, 26.33, 14.81, 30.94, 32.87, 33.31, 73.91, 73.92],
    'p_probability_max': [0.908, 0.900, 0.967, 0.931, 0.903, 0.932, 0.920, 0.912, 0.886, 0.862, 0.809, 0.953, 0.938, 0.954, 0.833, 0.959, 0.926, 0.885, 0.950, 0.845, 0.937],
    's_probability_max': [0.020, 0.570, 0.757, 0.073, 0.229, 0.256, 0.147, 0.160, 0.304, 0.470, 0.178, 0.064, 0.254, 0.212, 0.798, 0.767, 0.826, 0.630, 0.095, 0.045, 0.210]
}
df_ethz = pd.DataFrame(ethz_data)

# INSTANCE results
df_instance = pd.read_csv('/mnt/user-data/uploads/phasenet_picks_instance.csv')

# Merge
comparison = df_ethz.merge(df_instance, on='station_code', suffixes=('_ethz', '_instance'))

# Calculate differences
comparison['delta_t_p'] = comparison['t_p_seconds_instance'] - comparison['t_p_seconds_ethz']
comparison['delta_t_s'] = comparison['t_s_seconds_instance'] - comparison['t_s_seconds_ethz']
comparison['delta_ts_tp_ethz'] = comparison['t_s_seconds_ethz'] - comparison['t_p_seconds_ethz']
comparison['delta_ts_tp_instance'] = comparison['t_s_seconds_instance'] - comparison['t_p_seconds_instance']

# Identify problematic picks
comparison['s_before_p_ethz'] = comparison['delta_ts_tp_ethz'] < 0
comparison['s_before_p_instance'] = comparison['delta_ts_tp_instance'] < 0
comparison['ts_tp_unrealistic_ethz'] = np.abs(comparison['delta_ts_tp_ethz']) > 100
comparison['ts_tp_unrealistic_instance'] = np.abs(comparison['delta_ts_tp_instance']) > 100

print("=" * 80)
print("COMPARISON: PhaseNet ETHZ vs INSTANCE")
print("=" * 80)

print("\n### PROBLEMATIC PICKS ###\n")

print("ETHZ problematic stations:")
problematic_ethz = comparison[comparison['s_before_p_ethz'] | comparison['ts_tp_unrealistic_ethz']]
for _, row in problematic_ethz.iterrows():
    print(f"  {row['station_code']}: P={row['t_p_seconds_ethz']:.2f}s, S={row['t_s_seconds_ethz']:.2f}s, ΔT={row['delta_ts_tp_ethz']:.2f}s")

print(f"\nINSTANCE problematic stations:")
problematic_instance = comparison[comparison['s_before_p_instance'] | comparison['ts_tp_unrealistic_instance']]
if len(problematic_instance) == 0:
    print("  NONE - All picks are physically plausible!")
else:
    for _, row in problematic_instance.iterrows():
        print(f"  {row['station_code']}: P={row['t_p_seconds_instance']:.2f}s, S={row['t_s_seconds_instance']:.2f}s, ΔT={row['delta_ts_tp_instance']:.2f}s")

print("\n### ONSET TIME DIFFERENCES (INSTANCE - ETHZ) ###\n")

print(f"P-wave onset differences:")
print(f"  Mean: {comparison['delta_t_p'].mean():.3f} s")
print(f"  Std:  {comparison['delta_t_p'].std():.3f} s")
print(f"  Max absolute: {comparison['delta_t_p'].abs().max():.3f} s")

print(f"\nS-wave onset differences (excluding problematic ETHZ picks):")
valid_comparison = comparison[~(comparison['s_before_p_ethz'] | comparison['ts_tp_unrealistic_ethz'])]
print(f"  Mean: {valid_comparison['delta_t_s'].mean():.3f} s")
print(f"  Std:  {valid_comparison['delta_t_s'].std():.3f} s")
print(f"  Max absolute: {valid_comparison['delta_t_s'].abs().max():.3f} s")

print("\n### PROBABILITY IMPROVEMENTS ###\n")

print(f"S-wave probability improvements:")
comparison['s_prob_improvement'] = comparison['s_probability_max_instance'] - comparison['s_probability_max_ethz']
print(f"  Stations with improved S probability: {(comparison['s_prob_improvement'] > 0).sum()}/{len(comparison)}")
print(f"  Mean improvement: {comparison['s_prob_improvement'].mean():.3f}")

print("\n### SUMMARY ###\n")
print(f"ETHZ:")
print(f"  Stations processed: {len(df_ethz)}")
print(f"  Problematic picks: {len(problematic_ethz)} ({100*len(problematic_ethz)/len(df_ethz):.1f}%)")
print(f"  Valid picks: {len(df_ethz) - len(problematic_ethz)} ({100*(len(df_ethz)-len(problematic_ethz))/len(df_ethz):.1f}%)")

print(f"\nINSTANCE:")
print(f"  Stations processed: {len(df_instance)}")
print(f"  Problematic picks: {len(problematic_instance)} ({100*len(problematic_instance)/len(df_instance):.1f}%)")
print(f"  Valid picks: {len(df_instance) - len(problematic_instance)} ({100*(len(df_instance)-len(problematic_instance))/len(df_instance):.1f}%)")

print("\n### LARGEST CORRECTIONS BY INSTANCE ###\n")
print("\nStations where INSTANCE fixed ETHZ failures:")
fixed_stations = comparison[(comparison['s_before_p_ethz'] | comparison['ts_tp_unrealistic_ethz']) & 
                            ~(comparison['s_before_p_instance'] | comparison['ts_tp_unrealistic_instance'])]
for _, row in fixed_stations.iterrows():
    print(f"  {row['station_code']}:")
    print(f"    ETHZ:     P={row['t_p_seconds_ethz']:.2f}s, S={row['t_s_seconds_ethz']:.2f}s (ΔT={row['delta_ts_tp_ethz']:.2f}s)")
    print(f"    INSTANCE: P={row['t_p_seconds_instance']:.2f}s, S={row['t_s_seconds_instance']:.2f}s (ΔT={row['delta_ts_tp_instance']:.2f}s)")

## Code to copy and paste in terminal to commit & push on GitHub

cd ~/Desktop/PoliTo/Tesi/tesi-seismic-analysis && \
kaggle kernels pull giulianaparadiso/phase-picking-using-phasenet -p ./notebooks/ && \
git add notebooks/phase-picking-using-phasenet.ipynb && \
git commit -m "Update PhaseNet notebook from Kaggle" && \
git push origin main